## Load Taxi Data

In [6]:
import os
import time
import pandas as pd
from sodapy import Socrata
from datetime import date, timedelta

APP_TOKEN   = "xwqrqc987doRPgtIuryvxM2Ls"
DATASET_ID  = "ajtu-isnz"
OUTPUT_FILE = "../data/chicago_taxi_2025.csv"
CHECKPOINT_FILE = "../data/download_checkpoint_2025.txt"

COLUMNS = [
    "trip_start_timestamp",
    "trip_end_timestamp",
    "trip_seconds",
    "trip_miles",
    "pickup_census_tract",
    "dropoff_census_tract",
    "pickup_community_area",
    "dropoff_community_area",
    "pickup_centroid_latitude",
    "pickup_centroid_longitude",
    "dropoff_centroid_latitude",
    "dropoff_centroid_longitude",
    "fare",
    "trip_total",
]

CHUNK_DAYS  = 7      # Wochenchunks – klein genug für schnelle Serverantwort
BATCH_SIZE  = 50_000
MAX_RETRIES = 8
START_DATE  = date(2025, 1, 1)
END_DATE    = date(2025, 12, 31)


def load_checkpoint_date():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            d = date.fromisoformat(f.read().strip())
        print(f"  Checkpoint gefunden: Starte ab {d}")
        return d
    return START_DATE


def save_checkpoint_date(d):
    with open(CHECKPOINT_FILE, "w") as f:
        f.write(d.isoformat())


def append_to_csv(df, filepath):
    write_header = not os.path.exists(filepath)
    df.to_csv(filepath, mode="a", index=False, header=write_header)


def cast_types(df):
    df["trip_start_timestamp"] = pd.to_datetime(df["trip_start_timestamp"], errors="coerce")
    df["trip_end_timestamp"]   = pd.to_datetime(df["trip_end_timestamp"],   errors="coerce")
    for col in ["trip_seconds", "trip_miles", "fare", "trip_total",
                "pickup_centroid_latitude",  "pickup_centroid_longitude",
                "dropoff_centroid_latitude", "dropoff_centroid_longitude",
                "pickup_community_area",     "dropoff_community_area"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def download_taxi_2025():
    print("Verbinde mit Chicago Data Portal...")
    client = Socrata("data.cityofchicago.org", APP_TOKEN, timeout=180)

    current = load_checkpoint_date()
    select_clause = ", ".join(COLUMNS)
    total_rows = 0

    while current <= END_DATE:
        chunk_end = min(current + timedelta(days=CHUNK_DAYS), END_DATE + timedelta(days=1))
        where = (
            f"trip_start_timestamp >= '{ current.isoformat() }T00:00:00'"
            f" AND trip_start_timestamp < '{ chunk_end.isoformat() }T00:00:00'"
        )

        offset = 0
        while True:
            # Retry-Loop
            results = None
            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    results = client.get(
                        DATASET_ID,
                        select=select_clause,
                        where=where,
                        limit=BATCH_SIZE,
                        offset=offset,
                    )
                    break
                except Exception as e:
                    wait = 3 * (2 ** (attempt - 1))
                    print(f"  ✗ Versuch {attempt}/{MAX_RETRIES}: {e}")
                    if attempt < MAX_RETRIES:
                        print(f"    Warte {wait}s...")
                        time.sleep(wait)
                    else:
                        print(f"  Abbruch. Checkpoint bei {current}.")
                        save_checkpoint_date(current)
                        return

            if not results:
                break  # Kein Ergebnis in diesem Zeitfenster

            chunk = cast_types(pd.DataFrame.from_records(results))
            chunk = chunk[COLUMNS]
            append_to_csv(chunk, OUTPUT_FILE)

            offset     += len(results)
            total_rows += len(results)

            size_mb = os.path.getsize(OUTPUT_FILE) / 1e6
            print(f"  ✓ {current} – {chunk_end}  |  +{len(results):,} Zeilen  |  gesamt {total_rows:,}  |  {size_mb:.0f} MB")

            if len(results) < BATCH_SIZE:
                break  # Letzter Batch dieses Zeitfensters

        save_checkpoint_date(chunk_end)  # Nächste Woche beginnen
        current = chunk_end
        time.sleep(0.3)

    if os.path.exists(OUTPUT_FILE):
        final_size = os.path.getsize(OUTPUT_FILE) / 1e6
        print(f"── Fertig ─────────────────────────────────────────")
        print(f"  Datei  : {os.path.abspath(OUTPUT_FILE)}")
        print(f"  Groesse: {final_size:.1f} MB")
        print(f"  Zeilen : {total_rows:,}")
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)


download_taxi_2025()


Verbinde mit Chicago Data Portal...
  ✓ 2025-01-01 – 2025-01-08  |  +50,000 Zeilen  |  gesamt 50,000  |  6 MB
  ✓ 2025-01-01 – 2025-01-08  |  +32,779 Zeilen  |  gesamt 82,779  |  11 MB
  ✓ 2025-01-08 – 2025-01-15  |  +50,000 Zeilen  |  gesamt 132,779  |  17 MB
  ✓ 2025-01-08 – 2025-01-15  |  +46,088 Zeilen  |  gesamt 178,867  |  23 MB
  ✓ 2025-01-15 – 2025-01-22  |  +50,000 Zeilen  |  gesamt 228,867  |  30 MB
  ✓ 2025-01-15 – 2025-01-22  |  +43,995 Zeilen  |  gesamt 272,862  |  35 MB
  ✓ 2025-01-22 – 2025-01-29  |  +50,000 Zeilen  |  gesamt 322,862  |  42 MB
  ✓ 2025-01-22 – 2025-01-29  |  +50,000 Zeilen  |  gesamt 372,862  |  48 MB
  ✓ 2025-01-22 – 2025-01-29  |  +1,364 Zeilen  |  gesamt 374,226  |  49 MB
  ✓ 2025-01-29 – 2025-02-05  |  +50,000 Zeilen  |  gesamt 424,226  |  55 MB
  ✓ 2025-01-29 – 2025-02-05  |  +50,000 Zeilen  |  gesamt 474,226  |  62 MB
  ✓ 2025-01-29 – 2025-02-05  |  +3,302 Zeilen  |  gesamt 477,528  |  62 MB
  ✓ 2025-02-05 – 2025-02-12  |  +50,000 Zeilen  |  gesamt

## Load weather data

In [14]:
import requests
import pandas as pd

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude":  41.8781,   # Chicago
    "longitude": -87.6298,
    "start_date": "2025-01-01",
    "end_date":   "2025-12-31",
    "hourly": [
        "temperature_2m",
        "apparent_temperature",
        "precipitation",
        "snowfall",
        "wind_speed_10m",
        "cloud_cover",
        "weather_code",
    ],
    "timezone": "America/Chicago",  # lokale Zeit, passt zu den Taxidaten
    "wind_speed_unit": "mph",
}

response = requests.get(url, params=params)
data     = response.json()

# WMO Weather Code → human-readable description (Open-Meteo uses WMO standard)
WMO_CODES = {
    0:  "Clear sky",
    1:  "Mainly clear",
    2:  "Partly cloudy",
    3:  "Overcast", # bedeckt
    45: "Fog",
    48: "Rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",
    56: "Light freezing drizzle",
    57: "Heavy freezing drizzle",
    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",
    66: "Light freezing rain",
    67: "Heavy freezing rain",
    71: "Slight snowfall",
    73: "Moderate snowfall",
    75: "Heavy snowfall",
    77: "Snow grains",
    80: "Slight rain showers",
    81: "Moderate rain showers",
    82: "Violent rain showers",
    85: "Slight snow showers",
    86: "Heavy snow showers",
    95: "Thunderstorm",
    96: "Thunderstorm with slight hail",
    99: "Thunderstorm with heavy hail",
}

# In DataFrame umwandeln
df_weather = pd.DataFrame(data["hourly"])
df_weather["time"] = pd.to_datetime(df_weather["time"])
df_weather = df_weather.rename(columns={"time": "datetime"})

df_weather["weather_description"] = df_weather["weather_code"].map(WMO_CODES).fillna("Unknown")

print(f"Rows: {len(df_weather)}")   # should be 8760 (365 days × 24h)
print(df_weather[["datetime", "weather_code", "weather_description"]].head(10))

df_weather.to_csv("../data/chicago_weather_2025_hourly.csv", index=False)
print("Saved as ../data/chicago_weather_2025_hourly.csv")

Rows: 8760
             datetime  weather_code weather_description
0 2025-01-01 00:00:00             3            Overcast
1 2025-01-01 01:00:00             3            Overcast
2 2025-01-01 02:00:00             3            Overcast
3 2025-01-01 03:00:00            71     Slight snowfall
4 2025-01-01 04:00:00             3            Overcast
5 2025-01-01 05:00:00             3            Overcast
6 2025-01-01 06:00:00             3            Overcast
7 2025-01-01 07:00:00             3            Overcast
8 2025-01-01 08:00:00             0           Clear sky
9 2025-01-01 09:00:00             0           Clear sky
Saved as ../data/chicago_weather_2025_hourly.csv


## Data Cleaning – Taxi-Daten

Cleaning-Entscheidungen:
- **Census Tracts droppen**: Bei ~45 % der Chicago-internen Trips fehlt der Census Tract (Privacy-Masking). Community Area bietet dieselbe geografische Auflösung mit nahezu vollständiger Abdeckung.
- **Nur Trips innerhalb Chicagos**: Pickup *und* Dropoff müssen eine gültige Community Area haben. Außerhalb Chicagos fehlen Koordinaten fast durchgehend.
- **Fehlende Fahrtwerte**: Trips ohne `fare`, `trip_total`, `trip_seconds` oder `trip_miles` werden entfernt (abgebrochene / fehlerhafte Trips).
- **Ausreißer kappen**: Distanz und Dauer müssen > 0 sein; Kappen bei 99,9. Perzentile.

Ergebnis wird als **Parquet** gespeichert → kompakter und schneller ladbar als CSV.

In [1]:
import os
import pandas as pd

CSV_PATH    = '../data/chicago_taxi_2025.csv'
OUT_PARQUET = '../data/chicago_taxi_2025_clean.parquet'

# ── Laden ─────────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH, low_memory=False)
df['trip_start_timestamp'] = pd.to_datetime(df['trip_start_timestamp'], errors='coerce')
df['trip_end_timestamp']   = pd.to_datetime(df['trip_end_timestamp'],   errors='coerce')
df = df[df['trip_start_timestamp'].dt.year == 2025]
print(f'Raw:   {len(df):,} Zeilen, {df.shape[1]} Spalten')

# ── Schritt 1: Census Tracts droppen ─────────────────────────────────────────
df = df.drop(columns=['pickup_census_tract', 'dropoff_census_tract'])

# ── Schritt 2: Nur Trips innerhalb Chicagos (Pickup + Dropoff mit Community Area) ──
before = len(df)
df = df[df['pickup_community_area'].notna() & df['dropoff_community_area'].notna()]
print(f'Außerhalb Chicago entfernt:   {before - len(df):,}')
df['pickup_community_area']  = df['pickup_community_area'].astype(int)
df['dropoff_community_area'] = df['dropoff_community_area'].astype(int)

# ── Schritt 3: Fehlende Fahrtwerte entfernen ──────────────────────────────────
before = len(df)
df = df.dropna(subset=['fare', 'trip_total', 'trip_seconds', 'trip_miles'])
print(f'Fehlende Fahrtwerte entfernt: {before - len(df):,}')

# ── Schritt 4: Unmögliche Werte + Ausreißer (99,9. Perz.) entfernen ──────────
before = len(df)
df = df[(df['trip_miles'] > 0) & (df['trip_seconds'] > 0) & (df['fare'] > 0)]
cap_miles   = df['trip_miles'].quantile(0.999)
cap_seconds = df['trip_seconds'].quantile(0.999)
df = df[(df['trip_miles'] <= cap_miles) & (df['trip_seconds'] <= cap_seconds)]
print(f'Unmöglich/Ausreißer entfernt: {before - len(df):,}')
print(f'  Cap: {cap_miles:.1f} Meilen | {cap_seconds/60:.0f} Minuten')

# ── Speichern als Parquet ─────────────────────────────────────────────────────
df.to_parquet(OUT_PARQUET, index=False)
size_mb = os.path.getsize(OUT_PARQUET) / 1e6
print(f'\nGespeichert: {OUT_PARQUET}')
print(f'  Zeilen:  {len(df):,}')
print(f'  Spalten: {df.shape[1]}')
print(f'  Größe:   {size_mb:.1f} MB')

Raw:   6,825,826 Zeilen, 14 Spalten
Außerhalb Chicago entfernt:   651,117
Fehlende Fahrtwerte entfernt: 10,650
Unmöglich/Ausreißer entfernt: 455,795
  Cap: 33.2 Meilen | 150 Minuten

Gespeichert: ../data/chicago_taxi_2025_clean.parquet
  Zeilen:  5,708,264
  Spalten: 12
  Größe:   78.7 MB
